In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from app.course import Course
from app.database import CourseDatabase
from app.optimization import CourseOptimizer

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Course and Set Up Rider

We need a segmented course and rider parameters to optimize.

In [ ]:
# Load a course from database
db = CourseDatabase('../data/course_data.db')
available_courses = db.list_courses()

print("Available courses:")
for c in available_courses:
    print(f"  ID {c['id']}: {c['name']} ({c['course_type']})")

In [ ]:
# Choose course and load
COURSE_ID = 2  # Mt. Evans - more interesting terrain

course_info = db.get_course(COURSE_ID)
course_df = db.get_course_data(COURSE_ID)

# Create course and segment it
course = Course(course_df, skip_processing=True)
course.auto_segmentation(max_clusters=10)
aggregated_segments = course.aggregate_segments()

print(f"Course: {course_info['name']}")
print(f"Distance: {course_df['distance'].sum()/1000:.2f} km")
print(f"Segments: {len(course.course_segments)}")

In [ ]:
# Define rider parameters
RIDER_CONFIG = {
    'weight': 85,          # kg (rider + bike + gear)
    'critical_power': 266, # watts
    'w_prime': 20000,      # joules
    'cda': 0.32,           # m² (drops position)
    'crr': 0.004           # rolling resistance
}

print("Rider Configuration:")
for k, v in RIDER_CONFIG.items():
    print(f"  {k}: {v}")

## 2. Initialize the Optimizer

In [ ]:
# Create optimizer instance
optimizer = CourseOptimizer(
    course_segments=aggregated_segments,
    rider_weight=RIDER_CONFIG['weight'],
    critical_power=RIDER_CONFIG['critical_power'],
    w_prime=RIDER_CONFIG['w_prime'],
    cda=RIDER_CONFIG['cda'],
    crr=RIDER_CONFIG['crr']
)

print(f"Optimizer created with {len(optimizer.segments)} segments")
print(f"\nNormalized segments preview:")
optimizer.segments[['distance', 'gradient']].head()

## 3. Constant Power Optimization

Find the optimal steady-state power that uses W' effectively.

In [ ]:
# Optimize constant power
constant_result = optimizer.optimize_constant_power()

if constant_result['success']:
    print("Constant Power Optimization:")
    print(f"  Optimal power: {constant_result['optimal_power']:.1f} W")
    print(f"  Predicted time: {constant_result['total_time']:.1f} sec ({constant_result['total_time']/60:.2f} min)")
    print(f"  Final W' balance: {constant_result.get('final_w_prime', 'N/A')}")
else:
    print(f"Optimization failed: {constant_result.get('message', 'Unknown error')}")

In [ ]:
# Simulate a range of constant powers to see the time curve
test_powers = np.linspace(
    RIDER_CONFIG['critical_power'] * 0.9,
    RIDER_CONFIG['critical_power'] * 1.3,
    50
)

results = []
for power in test_powers:
    try:
        time = optimizer.simulate_course_time(np.full(len(optimizer.segments), power))
        results.append({'power': power, 'time': time, 'valid': True})
    except Exception as e:
        results.append({'power': power, 'time': None, 'valid': False})

results_df = pd.DataFrame(results)
valid_results = results_df[results_df['valid']]

In [ ]:
# Plot power vs time
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(valid_results['power'], valid_results['time']/60, 'b-', linewidth=2)

if constant_result['success']:
    ax.axvline(x=constant_result['optimal_power'], color='green', linestyle='--',
              label=f"Optimal: {constant_result['optimal_power']:.0f}W")
    ax.axhline(y=constant_result['total_time']/60, color='green', linestyle=':', alpha=0.5)

ax.axvline(x=RIDER_CONFIG['critical_power'], color='red', linestyle='--', alpha=0.5,
          label=f"CP: {RIDER_CONFIG['critical_power']}W")

ax.set_xlabel('Constant Power (W)')
ax.set_ylabel('Course Time (minutes)')
ax.set_title('Course Time vs Constant Power')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Variable Pacing Optimization

Optimize power per segment while managing W' depletion.

In [ ]:
# Variable pacing parameters
TARGET_W_PRIME_UTILIZATION = 0.7  # Use 70% of W'

variable_result = optimizer.optimize_variable_pacing(
    target_w_prime_utilization=TARGET_W_PRIME_UTILIZATION
)

if variable_result['success']:
    print("Variable Pacing Optimization:")
    print(f"  Predicted time: {variable_result['total_time']:.1f} sec ({variable_result['total_time']/60:.2f} min)")
    print(f"  Average power: {variable_result['average_power']:.1f} W")
    print(f"  Min W' balance: {variable_result.get('min_w_prime', 'N/A')}")
    print(f"\nSegment powers:")
    for i, power in enumerate(variable_result['segment_powers']):
        grad = optimizer.segments.iloc[i]['gradient'] * 100
        print(f"    Segment {i+1}: {power:.0f}W (gradient: {grad:+.1f}%)")
else:
    print(f"Optimization failed: {variable_result.get('message', 'Unknown error')}")

In [ ]:
# Compare constant vs variable pacing
if constant_result['success'] and variable_result['success']:
    time_savings = constant_result['total_time'] - variable_result['total_time']
    pct_savings = time_savings / constant_result['total_time'] * 100
    
    print("\n=== COMPARISON ===")
    print(f"Constant power time: {constant_result['total_time']/60:.2f} min")
    print(f"Variable pacing time: {variable_result['total_time']/60:.2f} min")
    print(f"Time savings: {time_savings:.1f} sec ({pct_savings:.1f}%)")

## 5. Visualize Pacing Strategies

In [ ]:
# Plot segment-by-segment comparison
if variable_result['success']:
    n_segments = len(optimizer.segments)
    segment_indices = np.arange(n_segments)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Power comparison
    bar_width = 0.35
    ax1.bar(segment_indices - bar_width/2, 
           [constant_result['optimal_power']] * n_segments,
           bar_width, label='Constant', alpha=0.7)
    ax1.bar(segment_indices + bar_width/2,
           variable_result['segment_powers'],
           bar_width, label='Variable', alpha=0.7)
    ax1.axhline(y=RIDER_CONFIG['critical_power'], color='red', linestyle='--',
               label=f"CP = {RIDER_CONFIG['critical_power']}W")
    ax1.set_ylabel('Power (W)')
    ax1.set_title('Power per Segment: Constant vs Variable Pacing')
    ax1.legend()
    
    # Gradient overlay
    ax2_twin = ax2.twinx()
    gradients = optimizer.segments['gradient'] * 100
    colors = ['green' if g >= 0 else 'red' for g in gradients]
    ax2.bar(segment_indices, gradients, color=colors, alpha=0.6)
    ax2.axhline(y=0, color='black', linewidth=0.5)
    ax2.set_ylabel('Gradient (%)')
    ax2.set_xlabel('Segment')
    
    # Distance on right axis
    distances = optimizer.segments['distance'] / 1000
    ax2_twin.plot(segment_indices, distances, 'ko-', label='Distance (km)')
    ax2_twin.set_ylabel('Distance (km)')
    
    ax2.set_title('Segment Characteristics')
    ax2.set_xticks(segment_indices)
    ax2.set_xticklabels([f'{i+1}' for i in segment_indices])
    
    plt.tight_layout()
    plt.show()

## 6. W' Depletion Profile

In [ ]:
def simulate_w_prime_profile(segments_df, segment_powers, cp, w_prime_max):
    """
    Simulate W' balance through the course.
    """
    w_prime = w_prime_max
    profile = [w_prime]
    cumulative_distance = [0]
    
    # Rough speed estimation for timing
    for i, (_, seg) in enumerate(segments_df.iterrows()):
        power = segment_powers[i]
        distance = seg['distance']
        
        # Estimate time for segment (simplified)
        avg_speed = 8  # m/s rough estimate
        time = distance / avg_speed
        
        # W' change
        if power > cp:
            w_prime -= (power - cp) * time
        else:
            # Recovery (simplified)
            w_prime += (cp - power) * time * 0.3
        
        w_prime = max(0, min(w_prime_max, w_prime))
        profile.append(w_prime)
        cumulative_distance.append(cumulative_distance[-1] + distance)
    
    return cumulative_distance, profile

if variable_result['success']:
    # Compare W' profiles
    const_powers = np.full(len(optimizer.segments), constant_result['optimal_power'])
    
    const_dist, const_wprime = simulate_w_prime_profile(
        optimizer.segments, const_powers,
        RIDER_CONFIG['critical_power'], RIDER_CONFIG['w_prime']
    )
    
    var_dist, var_wprime = simulate_w_prime_profile(
        optimizer.segments, variable_result['segment_powers'],
        RIDER_CONFIG['critical_power'], RIDER_CONFIG['w_prime']
    )

In [ ]:
if variable_result['success']:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    ax.plot(np.array(const_dist)/1000, np.array(const_wprime)/1000, 
           'b-', linewidth=2, label='Constant Power')
    ax.plot(np.array(var_dist)/1000, np.array(var_wprime)/1000,
           'g-', linewidth=2, label='Variable Pacing')
    
    ax.axhline(y=RIDER_CONFIG['w_prime']/1000, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(y=0, color='red', linewidth=2)
    
    # Target utilization line
    target_w = RIDER_CONFIG['w_prime'] * (1 - TARGET_W_PRIME_UTILIZATION) / 1000
    ax.axhline(y=target_w, color='orange', linestyle=':', 
              label=f'Target min ({(1-TARGET_W_PRIME_UTILIZATION)*100:.0f}% remaining)')
    
    ax.set_xlabel('Distance (km)')
    ax.set_ylabel('W\' Balance (kJ)')
    ax.set_title('W\' Depletion Profile: Constant vs Variable Pacing')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-1, RIDER_CONFIG['w_prime']/1000 * 1.1)
    
    plt.tight_layout()
    plt.show()

## 7. Parameter Sensitivity

How does target W' utilization affect results?

In [ ]:
# Test different W' utilization targets
utilization_targets = [0.3, 0.5, 0.7, 0.9]

sensitivity_results = []
for target in utilization_targets:
    try:
        result = optimizer.optimize_variable_pacing(target_w_prime_utilization=target)
        if result['success']:
            sensitivity_results.append({
                'target': target,
                'time': result['total_time'],
                'avg_power': result['average_power'],
                'powers': result['segment_powers']
            })
    except Exception as e:
        print(f"Target {target}: Failed - {e}")

print("W' Utilization Sensitivity:")
for r in sensitivity_results:
    print(f"  Target {r['target']*100:.0f}%: {r['time']/60:.2f} min, avg {r['avg_power']:.0f}W")

In [ ]:
if len(sensitivity_results) > 1:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    targets = [r['target']*100 for r in sensitivity_results]
    times = [r['time']/60 for r in sensitivity_results]
    powers = [r['avg_power'] for r in sensitivity_results]
    
    # Time vs utilization
    ax1.plot(targets, times, 'bo-', linewidth=2, markersize=10)
    ax1.set_xlabel('Target W\' Utilization (%)')
    ax1.set_ylabel('Course Time (minutes)')
    ax1.set_title('Effect of W\' Utilization Target on Time')
    ax1.grid(True, alpha=0.3)
    
    # Power vs utilization
    ax2.plot(targets, powers, 'go-', linewidth=2, markersize=10)
    ax2.axhline(y=RIDER_CONFIG['critical_power'], color='red', linestyle='--',
               label=f"CP = {RIDER_CONFIG['critical_power']}W")
    ax2.set_xlabel('Target W\' Utilization (%)')
    ax2.set_ylabel('Average Power (W)')
    ax2.set_title('Effect of W\' Utilization Target on Average Power')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. High-Resolution Simulation

Apply optimized powers to raw GPS points for accurate timing.

In [ ]:
# Get detailed simulation if the method exists
if hasattr(optimizer, 'simulate_course_detailed') and variable_result['success']:
    try:
        detailed = optimizer.simulate_course_detailed(
            segment_powers=variable_result['segment_powers'],
            course_data=course.processed_course
        )
        print(f"Detailed simulation completed")
        print(f"  Total points: {len(detailed)}")
        print(f"  Available columns: {list(detailed.columns)}")
    except Exception as e:
        print(f"Detailed simulation error: {e}")
        detailed = None
else:
    print("Detailed simulation not available")
    detailed = None

In [ ]:
if detailed is not None and 'cumulative_time' in detailed.columns:
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    cum_dist = detailed['cumulative_distance'] / 1000
    
    # Speed profile
    ax = axes[0]
    if 'speed' in detailed.columns:
        ax.plot(cum_dist, detailed['speed'] * 3.6, 'b-', linewidth=0.5)
        ax.set_ylabel('Speed (km/h)')
        ax.set_title('Speed Profile')
    
    # Power profile
    ax = axes[1]
    if 'power' in detailed.columns:
        ax.plot(cum_dist, detailed['power'], 'g-', linewidth=0.5)
        ax.axhline(y=RIDER_CONFIG['critical_power'], color='red', linestyle='--')
        ax.set_ylabel('Power (W)')
        ax.set_title('Power Profile')
    
    # W' balance
    ax = axes[2]
    if 'w_prime_balance' in detailed.columns:
        ax.fill_between(cum_dist, 0, detailed['w_prime_balance']/1000, alpha=0.3)
        ax.plot(cum_dist, detailed['w_prime_balance']/1000, 'b-')
        ax.set_ylabel('W\' Balance (kJ)')
        ax.set_xlabel('Distance (km)')
        ax.set_title('W\' Balance Through Course')
    
    plt.tight_layout()
    plt.show()
else:
    print("Detailed results not available for plotting")

## 9. Key Takeaways

- **Variable pacing** can save time by pushing harder on climbs and recovering on flats
- **W' utilization target** balances risk vs reward - higher targets are faster but riskier
- **High-resolution simulation** validates that optimized segments work on real terrain
- **Gradient-aware pacing** is key: gravity dominates on climbs, aero on flats